# Problem Statement

Customer Churn is a burning problem for Telecom companies. Almost every telecom company pays a premium to get a customer on-board. Customer churn is a directly impacts company’s revenue.
  
In this case-study, we simulate one such case of customer churn where we work on a data of post-paid customers with a contract. The data has information about customer usage behaviour, contract details, and payment details. The data also indicates which were the customers who cancelled their service.  
  
Based on this past data, Perform an EDA and build a model which can predict whether a customer will cancel their service in the future or not.

# Data Dictionary

* <b>Churn</b> - 1 if customer cancelled service, 0 if not
* <b>AccountWeeks</b> - number of weeks customer has had active account
* <b>ContractRenewal</b> - 1 if customer recently renewed contract, 0 if not
* <b>DataPlan</b> - 1 if customer has data plan, 0 if not
* <b>DataUsage</b> - gigabytes of monthly data usage
* <b>CustServCalls</b> - number of calls into customer service
* <b>DayMins</b> - average daytime minutes per month
* <b>DayCalls</b> - average number of daytime calls
* <b>MonthlyCharge</b> - average monthly bill
* <b>OverageFee</b> - largest overage fee in last 12 months
* <b>RoamMins</b> - average number of roaming minutes


In [1]:
#Import all necessary modules
import pandas as pd  
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn import metrics,model_selection
from sklearn.preprocessing import scale

In [3]:
cell_df = pd.read_excel(r'D:\My Data\Professional\Analytics\PGPDSBA - Other_Projects\Project 16 - Cellphone\Cellphone.xlsx')

cell_df.head()

,Churn,AccountWeeks,ContractRenewal,DataPlan,DataUsage,CustServCalls,DayMins,DayCalls,MonthlyCharge,OverageFee,RoamMins
0,1,108.0,0.0,0.0,0.38,2.0,115.1,114.0,41.8,10.57,13.8
1,0,113.0,1.0,1.0,3.59,1.0,44.9,63.0,55.9,6.71,13.3
2,0,101.0,1.0,0.0,0.00,0.0,257.3,84.0,60.0,9.24,13.5
3,0,80.0,1.0,1.0,2.89,4.0,166.4,92.0,77.9,11.92,10.7
4,0,70.0,1.0,0.0,0.00,0.0,175.4,130.0,44.0,7.98,11.6


## EDA

In [4]:
cell_df.head()

,Churn,AccountWeeks,ContractRenewal,DataPlan,DataUsage,CustServCalls,DayMins,DayCalls,MonthlyCharge,OverageFee,RoamMins
0,1,108.0,0.0,0.0,0.38,2.0,115.1,114.0,41.8,10.57,13.8
1,0,113.0,1.0,1.0,3.59,1.0,44.9,63.0,55.9,6.71,13.3
2,0,101.0,1.0,0.0,0.00,0.0,257.3,84.0,60.0,9.24,13.5
3,0,80.0,1.0,1.0,2.89,4.0,166.4,92.0,77.9,11.92,10.7
4,0,70.0,1.0,0.0,0.00,0.0,175.4,130.0,44.0,7.98,11.6


In [5]:
cell_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3333 entries, 0 to 3332
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Churn            3333 non-null   int64  
 1   AccountWeeks     3303 non-null   float64
 2   ContractRenewal  3315 non-null   float64
 3   DataPlan         3324 non-null   float64
 4   DataUsage        3317 non-null   float64
 5   CustServCalls    3281 non-null   float64
 6   DayMins          3298 non-null   float64
 7   DayCalls         3322 non-null   float64
 8   MonthlyCharge    3320 non-null   float64
 9   OverageFee       3309 non-null   float64
 10  RoamMins         3326 non-null   float64
dtypes: float64(10), int64(1)
memory usage: 286.6 KB


There are missing values in some columns.  
All variables are of numeric type and does not contain any data inconsistencies (causing numeric variables to be object due to some special characters present in the data).  
Churn is the target variable.   
Churn, ContractRenewal and DataPlan are binary variables.

In [6]:
cell_df[['AccountWeeks','DataUsage','CustServCalls','DayMins','DayCalls','MonthlyCharge','OverageFee','RoamMins']].describe()

,AccountWeeks,DataUsage,CustServCalls,DayMins,DayCalls,MonthlyCharge,OverageFee,RoamMins
count,3303.000000,3317.000000,3281.000000,3298.000000,3322.000000,3320.000000,3309.000000,3326.000000
mean,101.159552,0.814827,1.563852,179.863069,100.434377,56.288735,10.048598,10.237974
std,39.879736,1.270329,1.318590,54.581440,20.079248,16.438343,2.539063,2.793192
min,1.000000,0.000000,0.000000,0.000000,0.000000,14.000000,0.000000,0.000000
25%,74.000000,0.000000,1.000000,143.700000,87.000000,45.000000,8.330000,8.500000
50%,101.000000,0.000000,1.000000,179.600000,101.000000,53.500000,10.070000,10.300000
75%,127.000000,1.780000,2.000000,216.700000,114.000000,66.125000,11.770000,12.100000
max,243.000000,5.400000,9.000000,350.800000,165.000000,111.300000,18.190000,20.000000


### Check for Missing values

In [7]:
cell_df.isnull().sum()

Churn               0
AccountWeeks       30
ContractRenewal    18
DataPlan            9
DataUsage          16
CustServCalls      52
DayMins            35
DayCalls           11
MonthlyCharge      13
OverageFee         24
RoamMins            7
dtype: int64

### Imputing missing values

Since, ContractRenewal and DataPlan are binary, we cannot substitute with mean values for these 2 variables. We will impute these two variables with their respective modal values.

In [ ]:
# Compute Mode for the 'Contract Renewal' and 'DataPlan' columns and impute the missing values
cols = ['ContractRenewal','DataPlan']
for column in cols:
    
cell_df.isnull().sum()

Now let us impute the rest of the continuous variables with the median. For that we are going to use the SimpleImputer sub module from sklearn.

In [ ]:
from sklearn.impute import SimpleImputer
SI = SimpleImputer(strategy='median')

In [ ]:
#Now we need to fit and transform our respective data set to fill the missing values with the corresponding 'median' values



### Checking for Duplicates

In [ ]:
# Are there any duplicates ?
dups = cell_df.duplicated()
print('Number of duplicate rows = %d' % (dups.sum()))

### Proportion in the Target classes

In [ ]:
cell_df.Churn.value_counts(normalize=True)

### Distribution of the variables Check using Histogram

In [ ]:
cell_df[['AccountWeeks','DataUsage','CustServCalls','DayMins','DayCalls','MonthlyCharge','OverageFee','RoamMins']].hist();

### Outlier Check using boxplots

In [ ]:
cols=['AccountWeeks','DataUsage','CustServCalls','DayMins','DayCalls','MonthlyCharge','OverageFee','RoamMins'];

for i in cols:
    sns.boxplot(cell_df[i])
    plt.show()


### Bi-Variate Analysis with Target variable

<b>Account Weeks and Churn</b>

In [ ]:
sns.boxplot(cell_df['Churn'],cell_df['AccountWeeks'])

<b>Data Usage against Churn</b>

In [ ]:
sns.boxplot(cell_df['Churn'],cell_df['DataUsage'])

<b>DayMins against Churn</b>

In [ ]:
sns.boxplot(cell_df['Churn'],cell_df['DayMins'])

<b>DayCalls against Churn</b>

In [ ]:
sns.boxplot(cell_df['Churn'],cell_df['DayCalls'])

<b>MonthlyCharge against Churn</b>

In [ ]:
sns.boxplot(cell_df['Churn'],cell_df['MonthlyCharge'])

<b>OverageFee against Churn</b>

In [ ]:
sns.boxplot(cell_df['Churn'],cell_df['OverageFee'])

<b>RoamMins against Churn</b>

In [ ]:
sns.boxplot(cell_df['Churn'],cell_df['RoamMins'])

<b>CustServCalls against Churn</b>

In [ ]:
sns.boxplot(cell_df['Churn'],cell_df['CustServCalls'])

<b>Contract Renewal against Churn</b>

In [ ]:
sns.countplot(cell_df['ContractRenewal'],hue=cell_df['Churn'])

<b>Data Plan against Churn</b>

In [ ]:
sns.countplot(cell_df['DataPlan'],hue=cell_df['Churn'])

### Train (70%) - Test(30%) Split 

### LDA Model

In [ ]:
#Build LDA Model and fit the data





### Prediction and Evaluation on both Training and Test Set using Confusion Matrix, Classification Report and AUC-ROC.

In [ ]:
# Predict it




In [ ]:
# Evaluation



